In [11]:
from dotenv import load_dotenv
import os 
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import AzureChatOpenAI
from langchain_core.output_parsers import StrOutputParser

load_dotenv('env', override=True)
AZURE_OPENAI_API_KEY = os.getenv('AZURE_OPENAI_API_KEY')
END_POINT=os.getenv('END_POINT')
MODEL_NAME=os.getenv('MODEL_NAME')
print(AZURE_OPENAI_API_KEY[:10])
print(MODEL_NAME)

AZURE_OPENAI_EMB_API_KEY = os.getenv('AZURE_OPENAI_EMB_API_KEY')
EMB_END_POINT=os.getenv('EMB_END_POINT')
EMB_MODEL_NAME=os.getenv('EMB_MODEL_NAME')

os.environ['LANGCHAIN_API_KEY'] = os.getenv('LANGSMITH_API_KEY')
os.environ['LANGCHAIN_ENDPOINT'] = os.getenv('LANGCHAIN_ENDPOINT')
os.environ['LANGCHAIN_TRACING_V2'] = 'false' #true, false
os.environ['LANGCHAIN_PROJECT'] = 'LANG'

if os.getenv('LANGCHAIN_TRACING_V2') == "true":
    if len(os.getenv('LANGCHAIN_API_KEY')) > 0:
        print('랭스미스로 추적 중입니다 :', os.getenv('LANGSMITH_API_KEY')[:10])
    else:
        print('랭스미스 키가 확인되지 않았습니다.')

43b13g4OZS
gpt-5-mini


# Serialization

In [12]:
prompt = ChatPromptTemplate.from_template(
    "다음 내용을 아주 짧고 간단하게 한문장으로 설명해줘. 질문 :{question} 설명 : "
)

llm = AzureChatOpenAI(
    api_key=AZURE_OPENAI_API_KEY,
    azure_endpoint=END_POINT,  
    azure_deployment=MODEL_NAME,          
    api_version="2024-12-01-preview",
    temperature=0.2,
)

parser = StrOutputParser()

chain = prompt | llm | parser

print(chain.invoke({"question": "인공지능"}))

인공지능은 컴퓨터가 사람처럼 학습하고 판단하는 기술입니다.


In [13]:
# 에러 의도된것임
import pickle
with open("chain.pkl", "wb") as f:
    pickle.dump(chain, f)

TypeError: cannot pickle '_thread.RLock' object

In [14]:
print(chain.is_lc_serializable())

True


In [15]:
from langchain_core.load import dumpd, dumps #d는 dict, s는 json string
serialized_chain = dumpd(chain)

print(serialized_chain)

{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'schema', 'runnable', 'RunnableSequence'], 'kwargs': {'first': {'lc': 1, 'type': 'constructor', 'id': ['langchain', 'prompts', 'chat', 'ChatPromptTemplate'], 'kwargs': {'input_variables': ['question'], 'messages': [{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'prompts', 'chat', 'HumanMessagePromptTemplate'], 'kwargs': {'prompt': {'lc': 1, 'type': 'constructor', 'id': ['langchain', 'prompts', 'prompt', 'PromptTemplate'], 'kwargs': {'input_variables': ['question'], 'template': '다음 내용을 아주 짧고 간단하게 한문장으로 설명해줘. 질문 :{question} 설명 : ', 'template_format': 'f-string'}, 'name': 'PromptTemplate'}}}]}, 'name': 'ChatPromptTemplate'}, 'middle': [{'lc': 1, 'type': 'constructor', 'id': ['langchain', 'chat_models', 'azure_openai', 'AzureChatOpenAI'], 'kwargs': {'temperature': 0.2, 'openai_api_key': {'lc': 1, 'type': 'secret', 'id': ['AZURE_OPENAI_API_KEY']}, 'stream_usage': True, 'disabled_params': {'parallel_tool_calls': None}, 'output_versio

In [16]:
import json

with open("chain.json", "w") as f:
    json.dump(serialized_chain, f, ensure_ascii=False, indent=2)

In [17]:
import pickle

with open("chain.pkl", "wb") as f:
    pickle.dump(serialized_chain, f)

In [18]:
from langchain_core.load import load, loads

with open("chain.pkl", "rb") as f:
    loaded_chain = pickle.load(f)

loaded_chain = load(loaded_chain)

print(loaded_chain.invoke({"question": "인공지능"}))

/var/folders/jr/r6q5x5h966554q8401g1rk2m0000gn/T/ipykernel_66087/2847500803.py:6: LangChainBetaWarning: The function `load` is in beta. It is actively being worked on, so the API may change.
  loaded_chain = load(loaded_chain)


인공지능은 컴퓨터가 사람처럼 학습하고 판단하는 기술입니다.


In [21]:
with open("chain.json", "r") as f:
    loaded_chain = loads(f.read())
    
loaded_chain.invoke({"question": "머라이언"})

'머라이언은 싱가포르의 상징인 사자 머리와 물고기 몸통을 가진 조각상이다.'

In [22]:
load_chain = load(
    loaded_chain, secrets_map={"OPENAI_API_KEY": os.environ["AZURE_OPENAI_API_KEY"]}
)

# 불러온 체인이 정상 동작하는지 확인합니다.
load_chain.invoke({"question": "사과"})

'사과는 달고 아삭한 과일입니다.'